# Configurando ambiente spark

In [ ]:
# Instala a versão mais recente do PySpark (3.5+), que é 100% compatível com o Colab atual
!pip install -q pyspark pytest pytest-sugar wget findspark

  Preparing metadata (setup.py) ... done


In [ ]:
# import os

# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# os.environ["SPARK_HOME"] = "/content/spark-3.1.2-bin-hadoop2.7"

In [ ]:
# import findspark

# findspark.init()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
  .master('local[*]')\
  .appName("Iniciando com Spark")\
  .getOrCreate()

#Importe das libs e funções necessarias

In [ ]:
import wget
from pyspark.sql.functions import *
from pyspark.sql.window import Window
# from pyspark.sql.types import ArrayType, IntegerType

#Processo de extração dos dados

In [ ]:
# Nessa etapa foi criado uma pasta chamada "data" que irá armazenar
# os arquivos parquet e o arquivo csv. Os arquivos foram baixados usando o comando wget.
# com uma logica para pegar o arquivo de cada mes do ano de 2022

!mkdir './data'
!cd './data'

for month in range(1,13):
  if month < 10:
    month = '0'+str(month)
  wget.download(f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-{month}.parquet', f'./data/yellow_taxi_{month}-2022.parquet')
wget.download('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv', f'./data/taxi_zone_lookup.csv')


'./data/taxi_zone_lookup.csv'

In [ ]:
# Criação dos dataframes

df_taxis = spark.read.parquet('./data/yellow_taxi_*.parquet')
df_zones = spark.read.csv('./data/taxi_zone_lookup.csv', inferSchema=True, header=True, sep=',')


# Descrição das colunas do df_taxis e visualização das tabelas

**VendorID**: Um código indicando o provedor TPEP que forneceu o registro.

**tpep_pickup_datetime**: A data e hora em que o taxímetro foi acionado.

**tpep_dropoff_datetime**: A data e hora em que o taxímetro foi desligado.

**Passenger_count**: O número de passageiros no veículo.

**Trip_distance**: A distância da viagem em milhas relatada pelo taxímetro.

**PULocationID**: ID da Zona TLC na qual o taxímetro foi acionado.

**DOLocationID**: ID da Zona TLC na qual o taxímetro foi desligado.

**RateCodeID**: O código da tarifa em vigor no momento da viagem.
1 = Tarifa padrão;
2 = JFK;
3 = Newark;
4 = Nassau ou Westchester;
5 = Tarifa negociada;
6 = Grupo.

**Store_and_fwd_flag**: Este indicador mostra se o registro da viagem foi armazenado no veículo antes de ser enviado ao servidor.
Y = armazenar e encaminhar viagem
N = não armazenar e encaminhar viagem

**Payment_type**: Um código numérico que especifica como o passageiro pagou pela viagem.
1 = Crédito;
2 = Dinheiro;
3 = Sem cobrança;
4 = Disputa;
5 = Desconhecido;
6 = Viagem anulada (Voided trip).

**Fare_amount**: O valor da tarifa calculada pelo taxímetro.

**Extra**: Suplementos e encargos miscelâneos. Atualmente, isso inclui a sobretaxa de 0.50 à $1 para horas de pico e noturnas.

**MTA_tax**: Taxa de $0.50 que é automaticamente acionada pelo taxímetro.

**Improvement_surcharge**: Sobretaxa de $0.30 para viagens baseadas na taxa.

**Tip_amount**: O valor da gorjeta.

**Tolls_amount**: Total de pedágios pagos durante a viagem.

**Total_amount**: Total pago pelo passageiro.

**Congestion_Surcharge**: O total arrecadado pela taxa de congestionamento de NYC.

**Airport_fee**: $1.25 para retiradas nos aeroportos LaGuardia e John F. Kennedy.

In [ ]:
df_taxis.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [ ]:
df_taxis.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-10-01 00:03:41|  2022-10-01 00:18:39|            1.0|          1.7|       1.0|                 N|         249|         107|           1|        9.5|  3.0|    0.5|      2.6

In [ ]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [ ]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [ ]:
# @title
# Excluindo dados duplicados

df_taxis = df_taxis.dropDuplicates()

df_zones = df_zones.dropDuplicates()

In [ ]:
# @title
# Filtrando os dados para o ano de 2022
# E apenas as zonas yellow

# df_taxis = df_taxis.filter(col("Trip_distance") > 0)
# df_taxis = df_taxis.filter(col("Total_amount") >= 0)
df_taxis = df_taxis.filter(year(col("tpep_pickup_datetime")) == 2022)

df_zones = df_zones.filter(col("service_zone") == "Yellow Zone")

In [ ]:
# @title
# Definindo função para conferir se um dado é nulo ou NaN

def isnan_col(df, c):
    dtype = dict(df.dtypes)[c]
    if dtype in ['float', 'double', 'long']:
        return col(c).isNull() | col(c).isNaN()
    else:
        return col(c).isNull()

# 1-MTR (Substituição de Transformação de Mapeamento - Mapping Transformation Replacement)

In [ ]:
def mtr_function(df):
    return (
        df.withColumn(
            "trip_duration_min",
            (unix_timestamp(col("tpep_dropoff_datetime")) -
             unix_timestamp(col("tpep_pickup_datetime"))) / 60
        )
        # O coalesce garante que se a coluna for nula, ele assume 0.0 para a soma não quebrar
        .withColumn(
            "valor_calculado",
            coalesce(col("fare_amount"), lit(0.0)) +
            coalesce(col("extra"), lit(0.0)) +
            coalesce(col("mta_tax"), lit(0.0)) +
            coalesce(col("tip_amount"), lit(0.0)) +
            coalesce(col("tolls_amount"), lit(0.0)) +
            coalesce(col("improvement_surcharge"), lit(0.0)) +
            coalesce(col("congestion_surcharge"), lit(0.0)) +
            coalesce(col("airport_fee"), lit(0.0))
        )
        # O when previne o erro de divisão por zero
        .withColumn(
            "valor_por_km",
            when(
                col("trip_distance") > 0,
                col("fare_amount") / col("trip_distance")
            ).otherwise(lit(None))
        )
    )

# 2-NFTP (Negação do Predicado de Transformação de Filtragem - Negation of Filter Transformation Predicate)

In [ ]:
def nftp_function(df):
    return df.filter(
        col("trip_distance").isNotNull() &
        col("fare_amount").isNotNull() &
        (col("trip_distance") > 0.0) &
        (col("fare_amount") >= 2.50)
    )

# 3-ATR (Substituição de Transformação de Agregação - Aggregation Transformation Replacement)

In [ ]:
def atr_fuction(rdd):
    """
    Calcula a receita média por KM para cada motorista.
    Utiliza o padrão otimizado de MapReduce para evitar gargalos de memória (Out Of Memory).
    """

    # 1. MAP: Prepara os dados.
    # Transforma cada linha em: (Chave, (Valor1, Valor2))
    # Ex: (Motorista_1, (10.0 dolares, 2.0 km))
    pares_iniciais = rdd.map(
        lambda row: (row["VendorID"], (row["fare_amount"], row["trip_distance"]))
    )

    # 2. REDUCE: A Agregação Distribuída (O ALVO DO ATR)
    # Soma as posições correspondentes das tuplas: (Soma_Dolares, Soma_Km)
    # A função lambda pega o acumulador 'a' e o novo valor 'b' e soma índice com índice.
    totais_agregados = pares_iniciais.reduceByKey(
        lambda a, b: (a[0] + b[0], a[1] + b[1])
    )

    # 3. MAP FINAL: O Cálculo da Média
    # Divide o total de dólares pelo total de KM (com proteção contra divisão por zero)
    resultado_final = totais_agregados.mapValues(
        lambda totais: totais[0] / totais[1] if totais[1] > 0 else 0.0
    )

    return resultado_final

# 4-UTS (Troca de Transformações Unárias - Unary Transformations Swap)

In [ ]:
# %%writefile test_uts.py

# import pytest
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import col
# from pyspark.sql.types import StructType, StructField, StringType, DoubleType
# from pyspark.testing.utils import assertDataFrameEqual

# @pytest.fixture(scope="session")
# def spark():
#     # Inicialização limpa e moderna
#     spark_session = (SparkSession.builder
#         .master("local[1]")
#         .appName("pytest-pyspark-local")
#         .config("spark.ui.enabled", "false")
#         .getOrCreate())

#     yield spark_session
#     spark_session.stop()


# def uts_function(df):
#     """
#     1. Unary Transformation A: Filtra corridas com tarifa > 50.0
#     2. Unary Transformation B: Adiciona taxa administrativa de 5.0
#     """
#     return (
#         df
#         # Transformação 1 (Filter)
#         .filter(col("fare_amount") > 50.0)
#         # Transformação 2 (Map/withColumn)
#         .withColumn("fare_amount", col("fare_amount") + 5.0)
#     )


# def test_uts_function_inversao_de_sequencia(spark):
#     # 1. Preparação (Arrange)
#     schema = StructType([
#         StructField("trip_id", StringType(), True),
#         StructField("fare_amount", DoubleType(), True)
#     ])

#     input_data = [
#         # Original: 60.0 > 50 (Passa) -> 60.0 + 5.0 = 65.0
#         # Mutante: 60.0 + 5.0 = 65.0 -> 65.0 > 50 (Passa)
#         ("1", 60.0),

#         # Tarifa = 48.0
#         # Original: 48.0 > 50 (FALSO -> DESCARTADO)
#         # Mutante: 48.0 + 5.0 = 53.0 -> 53.0 > 50 (VERDADEIRO -> SOBREVIVE)
#         ("2", 48.0),

#         # Tarifa = 50.0
#         # Original: 50.0 > 50 (FALSO -> DESCARTADO)
#         # Mutante: 50.0 + 5.0 = 55.0 -> 55.0 > 50 (VERDADEIRO -> SOBREVIVE)
#         ("3", 50.0),

#         # Tarifa = 40.0
#         # Original: 40.0 > 50 (FALSO -> DESCARTADO)
#         # Mutante: 40.0 + 5.0 = 45.0 -> 45.0 > 50 (FALSO -> DESCARTADO)
#         ("4", 40.0)
#     ]

#     input_df = spark.createDataFrame(input_data, schema)

#     expected_data = [
#         ("1", 65.0)
#     ]

#     expected_df = spark.createDataFrame(expected_data, schema)

#     result_df = uts_function(input_df)

#     assertDataFrameEqual(result_df, expected_df)

Writing test_uts.py
